In [7]:
# Development version of AbstractAlgebra
using Pkg
Pkg.develop("AbstractAlgebra")

   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`


In [87]:
using Pkg, AbstractAlgebra
Pkg.status("AbstractAlgebra")

Status `~/.julia/environments/v1.12/Project.toml`
  [c3fe647b] AbstractAlgebra v0.50.1 `~/.julia/dev/AbstractAlgebra`


In [88]:
# Example ring
K, (t,) = puiseux_polynomial_ring(QQ, ["t"])

(Multivariate polynomial ring in 1 variable over multivariate Laurent polynomial ring, PuiseuxMPolyRingElem{Rational{BigInt}}[t])

This ring $K$ represents the Pusieux polynomial ring (over $\mathbb Q$):
$$
\bigcup_{n \in \mathbb N}\mathbb Q[t^{\frac 1n}] = 
\left\{
\sum_{i = 0}^{k} c_i t^{e_i} : 
c_i \in \mathbb Q,\ 
e_i \in \mathbb Q
\right\}
$$

In [89]:
f = t^(2//3) + 4*t^(1//3)

t^(2//3) + 4*t^(1//3)

In [90]:
# valuation is the 'min valuation'
println("val(f) = ", valuation(f))

val(f) = 1//3


In [91]:
g = t^(5//3) - t^(4//3);
println("f + g = ", f + g)
println("f * g = ", f * g)

f + g = t^(5//3) - t^(4//3) + t^(2//3) + 4*t^(1//3)
f * g = t^(7//3) + 3*t^2 - 4*t^(5//3)


In [92]:
# A few extra functions and Linear Algebra
include("PuiseuxRegression.jl");

In [93]:
h = 1 - t - t^2;
invf = inv_puiseux(h);
println("inv approx = ", invf)

inv approx = 8*t^5 + 5*t^4 + 3*t^3 + 2*t^2 + t + 1


In [94]:
# Drawing lines through points (y = c0 + c1*x + noise)

c0 = 2 + t^(1//2)
c1 = 3 - t^(1//3)

x1 = 1 + t
x2 = 2 + t^(2//3)
x3 = 3 + t^(1//4)

y1 = c0 + c1*x1 + t^5
y2 = c0 + c1*x2 + t^4
y3 = c0 + c1*x3 + t^6

D = [
    [x1, y1],
    [x2, y2],
    [x3, y3],
]

3-element Vector{Vector{PuiseuxMPolyRingElem{Rational{BigInt}}}}:
 [t + 1, t^5 - t^(4//3) + 3*t + t^(1//2) - t^(1//3) + 5]
 [t^(2//3) + 2, t^4 - t + 3*t^(2//3) + t^(1//2) - 2*t^(1//3) + 8]
 [t^(1//4) + 3, t^6 - t^(7//12) + t^(1//2) - 3*t^(1//3) + 3*t^(1//4) + 11]

In [95]:
# line through points 1 & 2 and 2 & 3

coef12 = dataPointsToFunction(D[[1,2]], truncNTerms=5)
coef23 = dataPointsToFunction(D[[2,3]], truncNTerms=5)

println("true coeffs = \n ", c0, "\n ", c1, "\n")
println("line: 1 & 2 = \n ", coef12[1], "\n ", coef12[2], "\n")
println("line: 1 & 2 = \n ", coef23[1], "\n ", coef23[2])

true coeffs = 
 t^(1//2) + 2
 -t^(1//3) + 3

line: 1 & 2 = 
 -25*t^5 + 20*t^(13//3) - 25*t^4 + t^(1//2) + 2
 t^5 - 20*t^(13//3) + 25*t^4 - t^(1//3) + 3

line: 1 & 2 = 
 -18*t^(19//12) + 6*t^(3//2) - 24*t^(17//12) + t^(1//2) + 2
 9*t^(19//12) - 3*t^(3//2) + 12*t^(17//12) - t^(1//3) + 3


In [96]:
# Loss vector: how far away (valuation) is each D[i] from the sampled line

println("lossVector(D, {1,2}) = ", lossVector(D, [1,2], Verbose=false))
println("lossVector(D, {2,3}) = ", lossVector(D, [1,3], Verbose=false))

lossVector(D, {1,2}) = Any[infinity, 4//1, 4//1]
lossVector(D, {2,3}) = Any[8//1, 5//2, 5//2]


In [97]:
# Generate a random set of 'true coefficients' for the linear function:
# f = c0 + c1*x

ctx = PContext(QQ, ["t"])
Kt = ctx.Kt
t = ctx.vars[1]

c = trueCoeffs(ctx, 2; L=2)
println("True coefficients:")
foreach(p -> println(p),c)

True coefficients:
1//5*t^5 - 5//9*t^4 + 1//7*t^3 - 1//7*t^2 + 1//8*t
-2//3*t^4 - 2*t^3 - 9*t^2 - 5//7*t
-9//2*t^(3//2) - 1//3*t


In [98]:
# Create a dataset y = c0 + c1*x1 + c2*x2 + noise
D = dataSet(c, 10; L=2, minShiftVal=1//1, maxShiftVal=3//1, truncNTerms=4)

println("First 3 generated points [x1, x2, y]:")
for i in 1:3
    println("D[$i] =")
    foreach(p -> println(" >> ", p), D[i])
    println()
end

First 3 generated points [x1, x2, y]:
D[1] =
 >> -7//4*t^4 - 1//7*t^(7//2) - 6*t^(5//2) + 5//7*t^(3//2) + 3//2*t
 >> 9//10*t^(67//16) - 7//6*t^(65//16) - 3//5*t^(49//16) - 9*t^(33//16) - 9//2*t^(1//16)
 >> -17//14*t^2 + 81//4*t^(25//16) + 3//2*t^(17//16) + 1//8*t

D[2] =
 >> -9*t^(113//32) - 1//10*t^(81//32) - 10//9*t^(49//32) - 5//3*t^(17//32) + 2//9*t^(1//2)
 >> 3*t^(29//8) - 2*t^(21//8) - 5*t^(13//8) + 5//2*t^(3//2) + 1//2*t^(1//2)
 >> -67//28*t^2 + 25//21*t^(49//32) - 41//126*t^(3//2) + 1//8*t

D[3] =
 >> 1//7*t^(23//8) + 4//7*t^(19//8) - t^(11//8) - 5//2*t^(9//8) - 2*t^(1//8)
 >> -2*t^(15//4) - 3//2*t^(5//2) - 7//9*t^(3//2) - 5*t
 >> 277//14*t^(17//8) + 32//21*t^2 + 10//7*t^(9//8) + 1//8*t



In [99]:
# 4) Fit using a subset S of size d+1 = 3
S = [1,2,3]
coef = dataPointsToFunction(D[S]; truncOutput=true, truncNTerms=5);
println("true coefs:")
foreach(p -> println(">> ", p),c)
println("")
println("line 1 & 2:")
foreach(p -> println(">> ", p), coef)

true coefs:
>> 1//5*t^5 - 5//9*t^4 + 1//7*t^3 - 1//7*t^2 + 1//8*t
>> -2//3*t^4 - 2*t^3 - 9*t^2 - 5//7*t
>> -9//2*t^(3//2) - 1//3*t

line 1 & 2:
>> -5//18*t^(59//32) - 5189853514601//55296*t^(29//16) + 2883251953125//2048*t^(57//32) - 192216796875//1024*t^(7//4) + 1//8*t
>> 1167717041007433//49152*t^(11//8) + 25949267583245//8192*t^(43//32) + 5189853514601//12288*t^(21//16) + 1729951171875//2048*t^(5//4) - 5//7*t
>> -1729951164707//1741824*t^(7//4) + 320361328125//1024*t^(55//32) - 21357421875//512*t^(27//16) - 9//2*t^(3//2) - 1//3*t


In [100]:
ls = loss(D, S; Verbose=false, truncOutput=true, truncNTerms=3)
lsvec = lossVector(D, S; Verbose=false, truncOutput=true, truncNTerms=3)

println("loss(D,S) = ", ls)
println("lossVector(D,S) = ", lsvec)

loss(D,S) = 11//8
lossVector(D,S) = Any[57//32, 29//16, 11//8, 7//4, 7//4, 7//4, 7//4, 7//4, 3//2, 7//4]
